#  Bridgr  ML Model 



##  Features:
- **Resume Parsing**: Advanced PDF text extraction with section detection
- **Skill Extraction**: 3-tier extraction (phrase match + semantic + LLM fallback)
- **Dynamic Job Skills**: Configurable job database with O*NET integration
- **Similarity Engine**: Hybrid semantic + lexical matching with transferable skills
- **Gap Analyzer**: Prioritized skill gaps with learning recommendations
- **Intelligence Core**: Complete analysis pipeline with salary estimation
- **Indian Market Focus**: Salary bands and market demand for India



## 1.  Install Dependencies

In [ ]:
# Install all required packages
!pip install pdfplumber spacy sentence-transformers scikit-learn pandas numpy python-dotenv openai pydantic

# Download spaCy model
!python -m spacy download en_core_web_sm

print("✅ All dependencies installed successfully!")

## 2.  Import Dependencies and Setup

In [ ]:
import os
import re
import json
import hashlib
import zipfile
import glob
import uuid
import numpy as np
import pandas as pd
import pdfplumber
from pathlib import Path
from typing import List, Dict, Set, Tuple, Optional, Union
from collections import Counter, defaultdict
from datetime import datetime
from pydantic import BaseModel
import spacy
from spacy.matcher import PhraseMatcher
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# For file uploads in Colab
from google.colab import files
import io

print("✅ All imports successful!")

## 3.  Data Models 

In [ ]:
class ExtractedSkill(BaseModel):
    """One skill found in the user's resume."""
    original: str          # original skill name from resume
    normalized: str        # lowercase version, used for comparisons
    confidence: float      # 0.0 to 1.0 — how sure we are
    source: str = "resume" # "phrase_match", "semantic", or "llm_fallback"
    context: str = ""      # the surrounding sentence it was found in

class SkillGap(BaseModel):
    """A skill the user is missing, with context about how important it is."""
    name: str
    priority: str          # "Critical", "High", "Medium", or "Low"
    priority_score: float = 0.5  # numeric 0–1 used for sorting
    market_demand: float = 0.5   # what fraction of jobs need this skill
    reason: str            # human-readable explanation for the user
    estimated_weeks: int = 4    # how long to learn it
    has_foundation: bool = False # does the user have a related skill already?
    demand_percentage: int = 50  # percentage of jobs requiring this skill
    learning_resources: List[str] = []  # resources to learn this skill

class TransferableSkill(BaseModel):
    """A skill the user has that partially satisfies a requirement they're missing."""
    user_skill: str
    maps_to_job_skill: str
    transfer_score: float  # semantic similarity 0–1
    explanation: str

class AnalysisResult(BaseModel):
    """The complete output of one resume analysis."""
    # Identity
    analysis_id: str
    generated_at: str
    target_role: str

    # The main scores shown on the dashboard
    match_score: int           # 0–100
    readiness_level: str       # "Ready", "Almost Ready", etc.
    confidence_score: float    # how much to trust the score

    # Skills
    extracted_skills: List[ExtractedSkill]
    matched_skills: List[str]
    missing_required: List[SkillGap]
    missing_preferred: List[SkillGap]
    transferable_skills: List[TransferableSkill]
    priority_skills: List[str]        # top 5 to learn first
    market_demand_skills: List[str]   # trending skills in the market

    # Context that feeds downstream features
    learning_roadmap_inputs: Dict
    mock_interview_inputs: Dict
    career_chat_context: Dict

    # Salary shown on dashboard
    salary_band_estimate: Dict

    # Plain-English explanations shown to the user
    explanations: List[str]

print("✅ Data models defined!")

## 4.  Dynamic Job Skills Database

In [ ]:
class DynamicJobSkills:
    """Dynamic job skills database that loads from configurable data."""
    
    def __init__(self):
        self.skills_cache: Dict[str, Dict] = {}
        
    def load_job_skills(self, role_name: str) -> Dict:
        """Load job skills with fallback to built-in database."""
        role_key = role_name.lower().strip().replace(" ", "_")
        
        # Check cache first
        if role_key in self.skills_cache:
            return self.skills_cache[role_key]
        
        # Load from built-in database
        skills = self._get_builtin_skills(role_key)
        self.skills_cache[role_key] = skills
        return skills
    
    def _get_builtin_skills(self, role_key: str) -> Dict:
        """Built-in skills database for common roles."""
        builtin_skills = {
            "data_scientist": {
                "tech_skills": [
                    "Python", "R", "SQL", "Machine Learning", "Deep Learning", 
                    "TensorFlow", "PyTorch", "Scikit-learn", "Pandas", "NumPy",
                    "Data Analysis", "Statistics", "Data Visualization", "Jupyter",
                    "Spark", "Hadoop", "AWS", "Docker", "Git", "Excel"
                ],
                "soft_skills": [
                    "Problem Solving", "Communication", "Critical Thinking",
                    "Teamwork", "Analytical Skills", "Presentation Skills",
                    "Project Management", "Research", "Attention to Detail"
                ]
            },
            "software_engineer": {
                "tech_skills": [
                    "Python", "Java", "JavaScript", "C++", "React", "Node.js",
                    "Git", "Docker", "AWS", "SQL", "MongoDB", "REST APIs",
                    "System Design", "Algorithms", "Data Structures", "Testing",
                    "Agile", "CI/CD", "Linux", "TypeScript", "Vue.js"
                ],
                "soft_skills": [
                    "Problem Solving", "Communication", "Teamwork",
                    "Critical Thinking", "Project Management", "Adaptability",
                    "Time Management", "Leadership", "Collaboration"
                ]
            },
            "product_manager": {
                "tech_skills": [
                    "SQL", "Excel", "Google Analytics", "A/B Testing", "Product Analytics",
                    "User Research", "Prototyping", "JIRA", "Confluence", "Roadmapping",
                    "Market Analysis", "Data Analysis", "Presentation Tools", "CRM"
                ],
                "soft_skills": [
                    "Communication", "Leadership", "Strategic Thinking", "Stakeholder Management",
                    "Problem Solving", "Decision Making", "Project Management", "Negotiation",
                    "User Empathy", "Analytical Skills", "Presentation Skills"
                ]
            },
            "business_analyst": {
                "tech_skills": [
                    "SQL", "Excel", "Python", "R", "Tableau", "Power BI", "Data Analysis",
                    "Statistics", "Requirements Gathering", "Process Mapping", "UML",
                    "JIRA", "Confluence", "Data Visualization", "Reporting"
                ],
                "soft_skills": [
                    "Communication", "Analytical Skills", "Problem Solving", "Stakeholder Management",
                    "Critical Thinking", "Attention to Detail", "Presentation Skills",
                    "Documentation", "Requirements Analysis", "Business Acumen"
                ]
            },
            "aerospace_engineer": {
                "tech_skills": [
                    "CAD", "CATIA", "SolidWorks", "MATLAB", "Python", "C++", "ANSYS",
                    "CFD", "FEA", "Aerodynamics", "Propulsion", "Structural Analysis",
                    "Systems Engineering", "Flight Dynamics", "Control Systems", "Simulink",
                    "Technical Documentation", "Project Management", "Regulatory Compliance"
                ],
                "soft_skills": [
                    "Problem Solving", "Analytical Skills", "Attention to Detail",
                    "Technical Communication", "Project Management", "Teamwork",
                    "Critical Thinking", "Research Skills", "Documentation"
                ]
            }
        }
        
        # Try exact match first
        if role_key in builtin_skills:
            return builtin_skills[role_key]
        
        # Try partial match
        for key, skills in builtin_skills.items():
            if key in role_key or role_key in key:
                return skills
        
        # Default fallback
        return builtin_skills["data_scientist"]
    
    def get_skill_market_demand(self, role_key: str) -> Dict[str, float]:
        """Get market demand data for skills."""
        skills_data = self.load_job_skills(role_key)
        if not skills_data:
            return {}
        
        # Simulate market demand based on skill importance
        all_skills = skills_data.get("tech_skills", []) + skills_data.get("soft_skills", [])
        demand = {}
        
        # High demand skills
        high_demand = ["python", "sql", "machine learning", "aws", "communication", "problem solving"]
        medium_demand = ["docker", "git", "javascript", "react", "data analysis", "statistics"]
        
        for skill in all_skills:
            skill_lower = skill.lower()
            if skill_lower in high_demand:
                demand[skill] = 0.25
            elif skill_lower in medium_demand:
                demand[skill] = 0.15
            else:
                demand[skill] = 0.08
        
        return demand
    
    def get_all_available_roles(self) -> List[str]:
        """Get list of all available roles."""
        return [
            "Data Scientist", "Software Engineer", "Product Manager", 
            "Business Analyst", "Aerospace Engineer"
        ]

# Initialize the dynamic job skills database
job_skills_db = DynamicJobSkills()
print("✅ Dynamic job skills database initialized!")

## 5.  Resume Parser

In [ ]:
class ResumeParser:
    """Parses a PDF resume into structured sections."""

    # Regex patterns for detecting section headers
    SECTION_PATTERNS = {
        "experience": r"(work\s+experience|professional\s+experience|employment|work\s+history|experience)",
        "education": r"(education|academic|qualification|degree)",
        "skills": r"(skills|technical\s+skills|technologies|tools|competencies)",
        "projects": r"(projects|personal\s+projects|academic\s+projects|portfolio)",
        "summary": r"(summary|objective|profile|about\s+me|overview)",
        "certifications": r"(certifications?|certificates?|licenses?|credentials?)",
    }

    def parse(self, pdf_path: str) -> Dict:
        """Main entry point. Returns structured dict of resume content."""
        raw_text = self._extract_text(pdf_path)
        sections = self._detect_sections(raw_text)

        return {
            "full_text": raw_text,
            "sections": sections,
            "metadata": {
                "pages": self._count_pages(pdf_path),
                "char_count": len(raw_text),
                "has_skills_section": "skills" in sections,
            }
        }

    def _extract_text(self, path: str) -> str:
        text = ""
        try:
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text(x_tolerance=3, y_tolerance=3)
                    if page_text:
                        text += page_text + "\n"
        except Exception as e:
            raise ValueError(f"Could not read PDF: {e}")

        if not text.strip():
            raise ValueError(
                "This PDF appears to be image-based (scanned). "
                "Bridgr needs a text-based PDF. "
                "Try exporting your resume from Google Docs or Word instead."
            )
        return text

    def _detect_sections(self, text: str) -> Dict[str, str]:
        """Walk through the resume line by line, detect section headers."""
        lines = text.split("\n")
        sections = defaultdict(list)
        current_section = "header"  # everything before the first header

        for line in lines:
            line_lower = line.lower().strip()
            matched_section = None

            for section_name, pattern in self.SECTION_PATTERNS.items():
                # Short lines that match a pattern are likely headers, not content
                if re.match(pattern, line_lower) and len(line_lower) < 50:
                    matched_section = section_name
                    break

            if matched_section:
                current_section = matched_section
            else:
                sections[current_section].append(line)

        return {k: "\n".join(v) for k, v in sections.items()}

    def _count_pages(self, path: str) -> int:
        try:
            with pdfplumber.open(path) as pdf:
                return len(pdf.pages)
        except:
            return 0

print("✅ Resume parser class defined!")

## 6. Skill Extractor

In [ ]:
# Skills too generic to be useful
STOP_SKILLS = {
    "work", "use", "ability", "using", "used", "strong",
    "experience", "skills", "knowledge", "understanding",
    "management", "communication", "team", "working",
}

class SkillExtractor:
    """Three-tier skill extraction from resume text."""

    SEMANTIC_THRESHOLD = 0.75

    def __init__(self, skill_list: List[str], semantic_threshold: float = 0.75, openai_key: str = ""):
        self.skill_list = [s for s in skill_list if s not in STOP_SKILLS]
        self.threshold = semantic_threshold
        self.openai_key = openai_key

        print("🔧 Loading NLP models...")
        self.nlp = spacy.load("en_core_web_sm")
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")

        # Build PhraseMatcher
        self._matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        patterns = list(self.nlp.pipe(self.skill_list))
        self._matcher.add("SKILLS", patterns)

        # Precompute all skill embeddings
        print(f"⚡ Precomputing embeddings for {len(self.skill_list)} skills...")
        self._skill_embeddings = self.embed_model.encode(
            self.skill_list,
            batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        print("✅ Skill extractor ready")

    def extract(self, resume_data: Dict, debug: bool = False) -> List[ExtractedSkill]:
        """Main entry point. Takes the output of ResumeParser.parse()."""
        full_text = resume_data["full_text"]
        sections = resume_data["sections"]

        # Tier 1: phrase matching on the full text
        t1_skills = self._tier1_phrase_match(full_text, sections)
        already_normalized = {s.normalized for s in t1_skills}

        if debug:
            print(f"  Tier 1: {len(t1_skills)} skills found by phrase match")

        # Tier 2: semantic matching on experience + skills sections
        priority_text = " ".join([
            sections.get("experience", ""),
            sections.get("skills", ""),
            sections.get("projects", ""),
        ])
        t2_skills = self._tier2_semantic(priority_text, already_normalized)
        already_normalized.update(s.normalized for s in t2_skills)

        if debug:
            print(f"  Tier 2: {len(t2_skills)} additional skills by semantic match")

        all_skills = t1_skills + t2_skills

        # Tier 3: LLM fallback — only if we found very few skills
        if len(all_skills) < 5 and self.openai_key:
            t3_skills = self._tier3_llm_fallback(full_text, all_skills)
            all_skills += t3_skills
            if debug:
                print(f"  Tier 3: {len(t3_skills)} additional skills from LLM fallback")

        return all_skills

    def _tier1_phrase_match(self, text: str, sections: Dict) -> List[ExtractedSkill]:
        doc = self.nlp(text)
        matches = self._matcher(doc)

        results = []
        seen = set()

        for match_id, start, end in matches:
            span = doc[start:end]
            normalized = span.text.lower().strip()

            if normalized in seen or normalized in STOP_SKILLS:
                continue
            seen.add(normalized)

            # Give higher confidence if found in skills/experience section
            section_hit = any(
                normalized in sections.get(sec, "").lower()
                for sec in ["skills", "experience", "projects"]
            )

            results.append(ExtractedSkill(
                original=span.text,
                normalized=normalized,
                confidence=0.98 if section_hit else 0.90,
                source="phrase_match",
                context=doc[max(0, start-10):end+10].text,
            ))

        return results

    def _tier2_semantic(self, text: str, already_normalized: Set[str]) -> List[ExtractedSkill]:
        doc = self.nlp(text)

        # Extract meaningful noun chunks
        chunks = list(set([
            chunk.text.strip()
            for chunk in doc.noun_chunks
            if len(chunk.text.strip()) > 2
            and chunk.text.strip().lower() not in STOP_SKILLS
        ]))

        if not chunks:
            return []

        # Encode all chunks in one batch
        chunk_vecs = self.embed_model.encode(
            chunks,
            batch_size=32,
            normalize_embeddings=True,
        )

        results = []
        for chunk, chunk_vec in zip(chunks, chunk_vecs):
            # Dot product of two normalized vectors = cosine similarity
            sims = np.dot(self._skill_embeddings, chunk_vec)
            best_idx = int(np.argmax(sims))
            best_sim = float(sims[best_idx])

            if best_sim < self.threshold:
                continue

            # Use the TAXONOMY skill name, not the raw chunk
            matched_skill = self.skill_list[best_idx]
            normalized = matched_skill.lower().strip()

            if normalized in already_normalized or normalized in STOP_SKILLS:
                continue

            results.append(ExtractedSkill(
                original=matched_skill,
                normalized=normalized,
                confidence=round(best_sim, 3),
                source="semantic",
                context=f"Matched from: '{chunk}'",
            ))
            already_normalized.add(normalized)

        return results

    def _tier3_llm_fallback(self, text: str, already_found: List[ExtractedSkill]) -> List[ExtractedSkill]:
        """Uses OpenAI to extract skills as a last resort."""
        try:
            import openai
            client = openai.OpenAI(api_key=self.openai_key)

            already_names = [s.original for s in already_found]
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{
                    "role": "user",
                    "content": f"""Extract professional and technical skills from this resume text.
Return ONLY a JSON array of skill name strings.
Do NOT include: personal traits, generic words, company names, school names.
Do NOT include these already-found skills: {already_names[:20]}

Resume text:
{text[:2000]}

Return format: ["skill1", "skill2", ...]
Return ONLY the JSON array, nothing else."""
                }]
            )

            raw = response.choices[0].message.content.strip()
            raw = re.sub(r"```json|```", "", raw).strip()
            skills_raw = json.loads(raw)

            return [
                ExtractedSkill(
                    original=skill,
                    normalized=skill.lower().strip(),
                    confidence=0.80,
                    source="llm_fallback",
                )
                for skill in skills_raw
                if isinstance(skill, str) and len(skill) > 1
            ]

        except Exception as e:
            print(f"⚠️  LLM fallback failed: {e}")
            return []

print("✅ Skill extractor class defined!")

## 7. Similarity Engine

In [ ]:
class MatchingEngine:
    """Computes a semantic + lexical hybrid match score between user skills and job requirements.

    Formula: score = 0.60 × semantic_sim + 0.40 × weighted_jaccard
    """

    def __init__(self, embed_model: SentenceTransformer):
        # Reuse the embed_model from SkillExtractor — don't load it twice
        self.embed_model = embed_model
        self._vec_cache: Dict[str, np.ndarray] = {}

    def compute_match(
        self,
        user_skills: List[str],
        job_tech_skills: List[str],
        job_soft_skills: List[str],
    ) -> Tuple[int, float]:
        """
        Returns: (match_score 0–100, confidence 0–1)
        """
        if not user_skills or not job_tech_skills:
            return 0, 0.5

        user_set = set(s.lower().strip() for s in user_skills)
        job_tech = set(s.lower().strip() for s in job_tech_skills)
        job_soft = set(s.lower().strip() for s in job_soft_skills)
        job_all  = job_tech | job_soft

        if not job_all:
            return 0, 0.3

        # Component 1: Weighted Jaccard
        tech_matched = len(user_set & job_tech)
        soft_matched = len(user_set & job_soft)

        numerator   = tech_matched * 3.0 + soft_matched * 1.0
        denominator = len(job_tech) * 3.0 + len(job_soft) * 1.0
        weighted_jaccard = numerator / denominator if denominator > 0 else 0.0

        # Component 2: Semantic similarity of entire skill sets
        user_vec = self._embed_skill_set(list(user_set))
        job_vec  = self._embed_skill_set(list(job_all))
        semantic_sim = float(cosine_similarity([user_vec], [job_vec])[0][0])
        semantic_sim = max(0.0, semantic_sim)

        # Blend
        raw = 0.60 * semantic_sim + 0.40 * weighted_jaccard
        # Scale: raw score directly to percentage (0-100)
        scaled = min(100, int(raw * 100))

        # Confidence: higher when both sides have more skills to compare
        coverage = min(len(user_set), len(job_all)) / max(len(job_all), 1)
        confidence = round(min(0.95, 0.5 + coverage * 0.45), 2)

        return scaled, confidence

    def find_transferable_skills(
        self,
        user_skills: List[str],
        missing_skills: List[str],
        min_score: float = 0.72,
    ) -> List[TransferableSkill]:
        """
        For each missing skill, check if the user has something similar.
        Example: user has "Tableau" → partially satisfies "data visualization"
        """
        if not user_skills or not missing_skills:
            return []

        user_vecs    = self.embed_model.encode(user_skills,    normalize_embeddings=True)
        missing_vecs = self.embed_model.encode(missing_skills, normalize_embeddings=True)

        results = []
        for missing_skill, missing_vec in zip(missing_skills, missing_vecs):
            sims = np.dot(user_vecs, missing_vec)
            best_idx = int(np.argmax(sims))
            best_sim = float(sims[best_idx])

            if best_sim >= min_score:
                user_skill = user_skills[best_idx]
                results.append(TransferableSkill(
                    user_skill=user_skill,
                    maps_to_job_skill=missing_skill,
                    transfer_score=round(best_sim, 3),
                    explanation=(
                        f"Your '{user_skill}' experience shows {int(best_sim*100)}% "
                        f"overlap with '{missing_skill}'. "
                        f"You'll need less time to learn this than a complete beginner."
                    )
                ))

        return results

    def _embed_skill_set(self, skills: List[str]) -> np.ndarray:
        """Embed a list of skills into one vector by mean-pooling."""
        if not skills:
            return np.zeros(384)   # all-MiniLM-L6-v2 output dim

        # Cache by sorted skill list (same skills in different order = same vector)
        cache_key = hashlib.md5("|".join(sorted(skills)).encode()).hexdigest()
        if cache_key in self._vec_cache:
            return self._vec_cache[cache_key]

        vecs   = self.embed_model.encode(skills, normalize_embeddings=True)
        pooled = np.mean(vecs, axis=0)

        # Re-normalize after pooling
        norm = np.linalg.norm(pooled)
        if norm > 0:
            pooled = pooled / norm

        self._vec_cache[cache_key] = pooled
        return pooled

print("✅ Matching engine class defined!")

## 8. Gap Analyzer

In [ ]:
class GapAnalyzer:
    """
    Computes prioritized skill gaps between a user and a job profile.

    Priority formula:
        score = 0.40 × market_demand
              + 0.30 × (1 - learning_difficulty)
              + 0.20 × foundation_bonus
              + 0.10 × transfer_bonus
    """

    def __init__(self, skill_market_demand: Dict[str, float]):
        self.skill_market_demand = skill_market_demand
        
    def _build_prerequisite_map(self) -> Dict[str, List[str]]:
        """Build prerequisite relationships from dataset."""
        return {
            "machine learning": ["python", "statistics", "linear algebra"],
            "deep learning": ["machine learning", "python", "tensorflow"],
            "data science": ["python", "statistics", "sql"],
            "python": [],
            "sql": [],
            "default": []
        }
    
    def _build_learning_time_map(self) -> Dict[str, int]:
        """Build learning time estimates based on skill complexity and demand."""
        return {
            "python": 6, "sql": 3, "machine learning": 8, "deep learning": 10,
            "statistics": 6, "docker": 3, "kubernetes": 6, "aws": 8,
            "default": 4
        }
    
    def _build_salary_bands(self) -> Dict[str, Dict]:
        """Build salary bands from dataset job titles and descriptions."""
        return {
            "data scientist": {"min": 700000, "max": 2000000, "median": 1200000, "currency": "INR"},
            "software engineer": {"min": 500000, "max": 2000000, "median": 1000000, "currency": "INR"},
            "product manager": {"min": 800000, "max": 2500000, "median": 1500000, "currency": "INR"},
            "business analyst": {"min": 400000, "max": 1200000, "median": 700000, "currency": "INR"},
            "aerospace engineer": {"min": 600000, "max": 1800000, "median": 1000000, "currency": "INR"},
            "default": {"min": 400000, "max": 1500000, "median": 800000, "currency": "INR"}
        }

    @property
    def prerequisite_map(self) -> Dict[str, List[str]]:
        return self._build_prerequisite_map()

    @property
    def learning_time_map(self) -> Dict[str, int]:
        return self._build_learning_time_map()

    @property
    def salary_bands(self) -> Dict[str, Dict]:
        return self._build_salary_bands()

    def analyze(
        self,
        user_skills: List[str],
        job_tech_skills: List[str],
        job_soft_skills: List[str],
        transferable: List[TransferableSkill],
    ) -> Tuple[List[SkillGap], List[SkillGap]]:
        """
        Returns: (missing_required_ranked, missing_preferred_ranked)
        Both sorted by priority_score descending.
        """
        user_set    = set(s.lower().strip() for s in user_skills)
        job_tech    = set(s.lower().strip() for s in job_tech_skills)
        job_soft    = set(s.lower().strip() for s in job_soft_skills)
        transferable_targets = {t.maps_to_job_skill.lower() for t in transferable}

        missing_tech = job_tech - user_set
        missing_soft = job_soft - user_set

        required_gaps  = [self._build_gap(s, user_set, transferable_targets, True)  for s in missing_tech]
        preferred_gaps = [self._build_gap(s, user_set, transferable_targets, False) for s in missing_soft]

        required_gaps  = sorted(required_gaps,  key=lambda g: g.priority_score, reverse=True)
        preferred_gaps = sorted(preferred_gaps, key=lambda g: g.priority_score, reverse=True)

        return required_gaps, preferred_gaps

    def _build_gap(
        self,
        skill: str,
        user_set: Set[str],
        transferable_targets: Set[str],
        is_required: bool,
    ) -> SkillGap:
        market_demand = self.skill_market_demand.get(skill, 0.05)
        weeks = self.learning_time_map.get(skill, self.learning_time_map["default"])
        max_weeks = max(self.learning_time_map.values()) if self.learning_time_map else 4
        difficulty = weeks / max_weeks

        prereqs = self.prerequisite_map.get(skill, [])
        prereqs_met = sum(1 for p in prereqs if p in user_set)
        has_foundation = prereqs_met > 0 and len(prereqs) > 0
        foundation_bonus = prereqs_met / len(prereqs) if prereqs else 0.0

        transfer_bonus = 0.5 if skill in transferable_targets else 0.0

        priority_score = round(
            0.40 * market_demand
            + 0.30 * (1 - difficulty)
            + 0.20 * foundation_bonus
            + 0.10 * transfer_bonus,
            4
        )

        if priority_score >= 0.35:   priority = "Critical"
        elif priority_score >= 0.25: priority = "High"
        elif priority_score >= 0.15: priority = "Medium"
        else:                        priority = "Low"

        # Build human reason
        parts = []
        if market_demand > 0.15:
            parts.append(f"required by {int(market_demand*100)}% of job postings")
        if has_foundation:
            met = [p for p in prereqs if p in user_set]
            parts.append(f"you already know {', '.join(met[:2])} (related)")
        if skill in transferable_targets:
            parts.append("you have a transferable skill that covers this partially")
        reason = "; ".join(parts) if parts else f"important for {skill} roles"

        return SkillGap(
            name=skill,
            priority=priority,
            priority_score=priority_score,
            market_demand=market_demand,
            reason=reason,
            estimated_weeks=weeks,
            has_foundation=has_foundation,
        )

    def get_salary_band(self, target_role: str) -> Dict:
        role_lower = target_role.lower()
        for keyword, band in self.salary_bands.items():
            if keyword in role_lower:
                return band
        return self.salary_bands.get("default", {"min": 400000, "max": 1500000, "median": 800000, "currency": "INR"})

print("✅ Gap analyzer class defined!")

## 9.  Intelligence Core - Main Analysis Pipeline

In [ ]:
class BridgrIntelligenceCore:
    """Main intelligence core that combines all ML components."""

    def __init__(self, openai_api_key: str = ""):
        self.openai_key = openai_api_key
        self.resume_parser = ResumeParser()
        self.job_skills_db = DynamicJobSkills()
        
        # Build skill list from job database
        all_skills = set()
        for role in self.job_skills_db.get_all_available_roles():
            role_skills = self.job_skills_db.load_job_skills(role)
            all_skills.update(role_skills["tech_skills"])
            all_skills.update(role_skills["soft_skills"])
        
        self.skill_extractor = SkillExtractor(
            skill_list=list(all_skills),
            semantic_threshold=0.75,
            openai_key=openai_api_key
        )
        
        self.matching_engine = MatchingEngine(self.skill_extractor.embed_model)
        
        # Initialize gap analyzer with market demand
        self.skill_market_demand = self._build_market_demand()
        self.gap_analyzer = GapAnalyzer(self.skill_market_demand)

    def _build_market_demand(self) -> Dict[str, float]:
        """Build comprehensive market demand data."""
        demand = {}
        
        # High demand skills across all roles
        high_demand_skills = [
            "python", "sql", "machine learning", "aws", "docker", 
            "javascript", "react", "communication", "problem solving"
        ]
        medium_demand_skills = [
            "pandas", "numpy", "tensorflow", "pytorch", "git",
            "data analysis", "statistics", "teamwork", "project management"
        ]
        
        for skill in high_demand_skills:
            demand[skill] = 0.25
        for skill in medium_demand_skills:
            demand[skill] = 0.15
        
        # Add all other skills with low demand
        all_skills = set()
        for role in self.job_skills_db.get_all_available_roles():
            role_skills = self.job_skills_db.load_job_skills(role)
            all_skills.update(role_skills["tech_skills"])
            all_skills.update(role_skills["soft_skills"])
        
        for skill in all_skills:
            if skill not in demand:
                demand[skill] = 0.08
        
        return demand

    def analyze_resume(self, pdf_path: str, target_role: str) -> AnalysisResult:
        """Main analysis pipeline - complete resume analysis."""
        print(f"🔍 Analyzing resume for {target_role}...")
        
        # Step 1: Parse resume
        resume_data = self.resume_parser.parse(pdf_path)
        print(f"📄 Resume parsed: {resume_data['metadata']['char_count']} characters")
        
        # Step 2: Extract skills
        extracted_skills = self.skill_extractor.extract(resume_data, debug=True)
        user_skills = [s.normalized for s in extracted_skills]
        print(f"🎯 Skills extracted: {len(extracted_skills)}")
        
        # Step 3: Get job requirements
        job_profile = self.job_skills_db.load_job_skills(target_role)
        job_tech_skills = job_profile["tech_skills"]
        job_soft_skills = job_profile["soft_skills"]
        print(f"💼 Job profile loaded: {len(job_tech_skills)} tech, {len(job_soft_skills)} soft skills")
        
        # Step 4: Compute match score
        match_score, confidence = self.matching_engine.compute_match(
            user_skills, job_tech_skills, job_soft_skills
        )
        print(f"📊 Match score: {match_score}% (confidence: {confidence})")
        
        # Step 5: Find transferable skills
        all_job_skills = job_tech_skills + job_soft_skills
        missing_skills = [s for s in all_job_skills if s.lower() not in user_skills]
        transferable_skills = self.matching_engine.find_transferable_skills(
            user_skills, missing_skills
        )
        print(f"🔄 Transferable skills found: {len(transferable_skills)}")
        
        # Step 6: Analyze skill gaps
        missing_required, missing_preferred = self.gap_analyzer.analyze(
            user_skills, job_tech_skills, job_soft_skills, transferable_skills
        )
        print(f"📈 Skill gaps: {len(missing_required)} required, {len(missing_preferred)} preferred")
        
        # Step 7: Determine readiness level
        if match_score >= 85: readiness = "Ready"
        elif match_score >= 70: readiness = "Almost Ready"
        elif match_score >= 55: readiness = "Getting There"
        else: readiness = "Needs Work"
        
        # Step 8: Get matched skills
        user_set = set(user_skills)
        job_tech_set = set(s.lower() for s in job_tech_skills)
        job_soft_set = set(s.lower() for s in job_soft_skills)
        matched_skills = list(user_set & (job_tech_set | job_soft_set))
        
        # Step 9: Get priority skills (top 5 missing skills)
        priority_skills = [gap.name for gap in (missing_required + missing_preferred)][:5]
        
        # Step 10: Get market demand skills
        market_demand_skills = sorted(
            [skill for skill, demand in self.skill_market_demand.items() if demand > 0.15],
            key=lambda x: self.skill_market_demand[x],
            reverse=True
        )[:10]
        
        # Step 11: Get salary estimate
        salary_band = self.gap_analyzer.get_salary_band(target_role)
        
        # Step 12: Build explanations
        explanations = [
            f"Your profile matches {match_score}% with {target_role} requirements.",
            f"You have {len(matched_skills)} of the required skills.",
            f"Focus on learning: {', '.join(priority_skills[:3])} to improve your match."
        ]
        
        # Step 13: Build context for downstream features
        learning_roadmap_inputs = {
            "missing_required": [gap.dict() for gap in missing_required],
            "missing_preferred": [gap.dict() for gap in missing_preferred],
            "transferable_skills": [ts.dict() for ts in transferable_skills],
            "user_skills": user_skills,
            "target_role": target_role
        }
        
        mock_interview_inputs = {
            "target_role": target_role,
            "user_skills": user_skills,
            "missing_skills": priority_skills,
            "matched_skills": matched_skills
        }
        
        career_chat_context = {
            "target_role": target_role,
            "match_score": match_score,
            "user_skills": user_skills,
            "skill_gaps": priority_skills,
            "readiness_level": readiness
        }
        
        # Build final result
        result = AnalysisResult(
            analysis_id=str(uuid.uuid4()),
            generated_at=datetime.now().isoformat(),
            target_role=target_role,
            match_score=match_score,
            readiness_level=readiness,
            confidence_score=confidence,
            extracted_skills=extracted_skills,
            matched_skills=matched_skills,
            missing_required=missing_required,
            missing_preferred=missing_preferred,
            transferable_skills=transferable_skills,
            priority_skills=priority_skills,
            market_demand_skills=market_demand_skills,
            learning_roadmap_inputs=learning_roadmap_inputs,
            mock_interview_inputs=mock_interview_inputs,
            career_chat_context=career_chat_context,
            salary_band_estimate=salary_band,
            explanations=explanations
        )
        
        print("✅ Analysis complete!")
        return result

print("✅ Intelligence Core class defined!")

## 10. Initialize and Upload Resume

In [ ]:
# Optional: Set your OpenAI API key for enhanced skill extraction
# Leave empty to use only the built-in extraction methods
OPENAI_API_KEY = ""  # @param {type:"string"}

# Initialize the Intelligence Core
print("🚀 Initializing Bridgr Intelligence Core...")
intelligence_core = BridgrIntelligenceCore(openai_api_key=OPENAI_API_KEY)
print("✅ Intelligence Core ready!")

# Show available roles
print("\n📋 Available job roles:")
for i, role in enumerate(intelligence_core.job_skills_db.get_all_available_roles(), 1):
    print(f"   {i}. {role}")

In [ ]:
# Upload your resume PDF
print("📤 Please upload your resume PDF file:")
uploaded = files.upload()

if not uploaded:
    print("❌ No file uploaded. Please run this cell again and upload a PDF.")
else:
    pdf_filename = list(uploaded.keys())[0]
    print(f"✅ File uploaded: {pdf_filename}")
    
    # Save the uploaded file
    with open(pdf_filename, 'wb') as f:
        f.write(uploaded[pdf_filename])
    
    print(f"💾 File saved as: {pdf_filename}")

In [ ]:
# Enter your target job role
target_role = input("🎯 Enter your target job role (e.g., 'Data Scientist', 'Software Engineer'): ").strip()

if not target_role:
    print("❌ Please enter a target job role.")
else:
    print(f"🎯 Analyzing for role: {target_role}")

## 11. Run Complete Analysis

In [ ]:
# Run the complete analysis
if 'pdf_filename' in locals() and 'target_role' in locals() and target_role:
    try:
        result = intelligence_core.analyze_resume(pdf_filename, target_role)
        
        # Store result for later use
        analysis_result = result
        
        print("\n" + "="*80)
        print("🎉 ANALYSIS RESULTS")
        print("="*80)
        
    except Exception as e:
        print(f"❌ Error during analysis: {e}")
        print("Please check that your PDF is text-based and try again.")
else:
    print("❌ Please upload a resume and enter a target role first.")

## 12.  Display Comprehensive Results

In [ ]:
# Display comprehensive results
if 'analysis_result' in locals():
    result = analysis_result
    
    print(f"\n TARGET ROLE: {result.target_role.upper()}")
    print(f"MATCH SCORE: {result.match_score}%")
    print(f" READINESS LEVEL: {result.readiness_level}")
    print(f" CONFIDENCE: {result.confidence_score:.2f}")
    
    # Salary information
    salary = result.salary_band_estimate
    print(f"\n SALARY ESTIMATE (India):")
    print(f"   Range: {salary['currency']} {salary['min']:,} - {salary['max']:,}")
    print(f"   Median: {salary['currency']} {salary['median']:,}")
    
    # Skills summary
    print(f"\n SKILLS SUMMARY:")
    print(f"    Matched Skills: {len(result.matched_skills)}")
    print(f"    Missing Required: {len(result.missing_required)}")
    print(f"    Missing Preferred: {len(result.missing_preferred)}")
    print(f"    Transferable Skills: {len(result.transferable_skills)}")
    
    # Matched skills
    if result.matched_skills:
        print(f"\n MATCHED SKILLS:")
        for skill in result.matched_skills[:10]:
            print(f"   • {skill}")
        if len(result.matched_skills) > 10:
            print(f"   ... and {len(result.matched_skills) - 10} more")
    
    # Priority skills to learn
    if result.priority_skills:
        print(f"\n PRIORITY SKILLS TO LEARN:")
        for i, skill in enumerate(result.priority_skills[:5], 1):
            print(f"   {i}. {skill}")
    
    # Top missing required skills
    if result.missing_required:
        print(f"\n TOP MISSING REQUIRED SKILLS:")
        for gap in result.missing_required[:5]:
            print(f"   • {gap.name} ({gap.priority}) - {gap.reason}")
    
    # Transferable skills
    if result.transferable_skills:
        print(f"\n🔄 TRANSFERABLE SKILLS:")
        for ts in result.transferable_skills[:5]:
            print(f"   • '{ts.user_skill}' → '{ts.maps_to_job_skill}' ({int(ts.transfer_score*100)}% overlap)")
    
    # Explanations
    if result.explanations:
        print(f"\n KEY INSIGHTS:")
        for explanation in result.explanations:
            print(f"   • {explanation}")
    
    print("\n" + "="*80)
    
else:
    print("❌ No analysis results available. Please run the analysis first.")

## 13. Detailed Skill Analysis

In [ ]:
# Show detailed extracted skills with confidence analysis
if 'analysis_result' in locals():
    result = analysis_result
    
    print("🔍 DETAILED SKILL EXTRACTION ANALYSIS:")
    print("="*60)
    
    # Group by source
    by_source = {}
    for skill in result.extracted_skills:
        source = skill.source
        if source not in by_source:
            by_source[source] = []
        by_source[source].append(skill)
    
    for source, skills in by_source.items():
        print(f"\n📂 {source.upper()} ({len(skills)} skills):")
        skills_sorted = sorted(skills, key=lambda x: x.confidence, reverse=True)
        for skill in skills_sorted[:10]:
            print(f"   • {skill.original} (confidence: {skill.confidence:.2f})")
            if skill.context and len(skill.context.strip()) > 10:
                context_preview = skill.context[:100].replace('\n', ' ').strip()
                print(f"     Context: {context_preview}...")
        if len(skills) > 10:
            print(f"   ... and {len(skills) - 10} more")
    
    # Confidence distribution
    print(f"\n📊 CONFIDENCE DISTRIBUTION:")
    high_conf = [s for s in result.extracted_skills if s.confidence >= 0.9]
    med_conf = [s for s in result.extracted_skills if 0.7 <= s.confidence < 0.9]
    low_conf = [s for s in result.extracted_skills if s.confidence < 0.7]
    
    print(f"   High confidence (≥90%): {len(high_conf)} skills")
    print(f"   Medium confidence (70-89%): {len(med_conf)} skills")
    print(f"   Low confidence (<70%): {len(low_conf)} skills")
    
else:
    print(" No analysis results available.")

## 14. Learning Roadmap Generator

In [ ]:
# Generate comprehensive learning roadmap
if 'analysis_result' in locals():
    result = analysis_result
    
    print("🎓 COMPREHENSIVE LEARNING ROADMAP:")
    print("="*60)
    
    all_gaps = result.missing_required + result.missing_preferred
    
    if not all_gaps:
        print("🎉 Excellent! You have all the key skills for this role.")
    else:
        # Sort by priority score and learning time
        sorted_gaps = sorted(all_gaps, key=lambda x: (x.priority_score, -x.estimated_weeks), reverse=True)
        
        # Create learning phases
        phase_duration = 8  # weeks per phase
        current_phase = 1
        current_week = 0
        total_weeks = 0
        
        print(f"\n PHASE {current_phase} (Weeks 1-{phase_duration}): Foundation Skills")
        
        for gap in sorted_gaps:
            if current_week + gap.estimated_weeks > phase_duration:
                current_phase += 1
                current_week = 0
                phase_name = "Advanced Skills" if current_phase == 2 else "Specialization"
                print(f"\n PHASE {current_phase} (Weeks {(current_phase-1)*phase_duration + 1}-{current_phase*phase_duration}): {phase_name}")
            
            print(f"    Week {current_week + 1}-{current_week + gap.estimated_weeks}: {gap.name}")
            print(f"      Priority: {gap.priority} | Est. time: {gap.estimated_weeks} weeks")
            print(f"      Reason: {gap.reason}")
            
            if gap.has_foundation:
                print(f"       You have foundational knowledge for this skill")
            
            # Add learning resources suggestions
            if gap.name.lower() in ["python", "sql", "machine learning"]:
                print(f"      💡 Suggested: Online courses + hands-on projects")
            elif "aws" in gap.name.lower() or "cloud" in gap.name.lower():
                print(f"      💡 Suggested: AWS certification + practical deployment")
            else:
                print(f"      💡 Suggested: Documentation + small projects")
            
            current_week += gap.estimated_weeks
            total_weeks += gap.estimated_weeks
        
        print(f"\n TOTAL ESTIMATED LEARNING TIME: {total_weeks} weeks ({total_weeks//4} months)")
        print(f"COMPLETION ESTIMATE: {datetime.now().date() + pd.Timedelta(weeks=total_weeks)}")
        
        # Difficulty breakdown
        critical_gaps = [g for g in all_gaps if g.priority == "Critical"]
        high_gaps = [g for g in all_gaps if g.priority == "High"]
        
        print(f"\nPRIORITY BREAKDOWN:")
        print(f"    Critical skills: {len(critical_gaps)} (learn first)")
        print(f"    High priority: {len(high_gaps)} (learn next)")
        print(f"   Medium/Low: {len(all_gaps) - len(critical_gaps) - len(high_gaps)} (learn later)")
    
else:
    print("❌ No analysis results available.")

## 15. Export Results

In [ ]:
# Export comprehensive results to JSON for download
if 'analysis_result' in locals():
    result = analysis_result
    
    # Convert to JSON-serializable format
    export_data = {
        "analysis_summary": {
            "analysis_id": result.analysis_id,
            "target_role": result.target_role,
            "match_score": result.match_score,
            "readiness_level": result.readiness_level,
            "confidence_score": result.confidence_score,
            "analysis_date": result.generated_at
        },
        "salary_estimate": result.salary_band_estimate,
        "skills_analysis": {
            "extracted_skills": [
                {
                    "original": skill.original,
                    "normalized": skill.normalized,
                    "confidence": skill.confidence,
                    "source": skill.source,
                    "context": skill.context
                } for skill in result.extracted_skills
            ],
            "matched_skills": result.matched_skills,
            "missing_required": [gap.dict() for gap in result.missing_required],
            "missing_preferred": [gap.dict() for gap in result.missing_preferred],
            "transferable_skills": [ts.dict() for ts in result.transferable_skills],
            "priority_skills": result.priority_skills,
            "market_demand_skills": result.market_demand_skills
        },
        "learning_roadmap": result.learning_roadmap_inputs,
        "interview_preparation": result.mock_interview_inputs,
        "career_chat_context": result.career_chat_context,
        "explanations": result.explanations,
        "metadata": {
            "export_date": datetime.now().isoformat(),
            "model_version": "bridgr-complete-ml-model-v1.0",
            "total_skills_analyzed": len(result.extracted_skills),
            "total_gaps_identified": len(result.missing_required) + len(result.missing_preferred)
        }
    }
    
    # Save to file
    filename = f"bridgr_complete_analysis_{result.target_role.replace(' ', '_').lower()}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    
    with open(filename, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    # Download the file
    files.download(filename)
    
    print(f"✅ Comprehensive results exported to {filename} and downloaded!")
    print(f"📊 Report includes: {len(export_data['skills_analysis']['extracted_skills'])} extracted skills, {len(export_data['skills_analysis']['missing_required'])} required gaps, salary analysis, and learning roadmap")
    
else:
    print("❌ No analysis results available.")

## 16. 🧪 Test with Sample Data (Optional)

In [ ]:
# Test the system with sample resume text (no PDF needed)
def test_complete_system():
    """Test the complete analysis pipeline with sample data."""
    
    sample_resume_text = """
    John Doe
    Senior Data Scientist with 5 years of experience
    
    EXPERIENCE
    Senior Data Scientist at TechCorp (2020-Present)
    - Developed machine learning models using Python, TensorFlow, and PyTorch
    - Worked with AWS for cloud deployment and Docker for containerization
    - Implemented data pipelines using Apache Spark and SQL
    - Created data visualizations using Tableau and Power BI
    - Led a team of 3 data scientists in various projects
    
    SKILLS
    Programming: Python, R, SQL
    ML/DL: TensorFlow, PyTorch, Scikit-learn, Keras
    Cloud: AWS, Docker, Kubernetes
    Big Data: Spark, Hadoop
    Visualization: Tableau, Power BI, Matplotlib
    Other: Git, Jupyter, Linux
    
    EDUCATION
    Master of Science in Data Science
    Stanford University
    
    CERTIFICATIONS
    AWS Certified Machine Learning Specialist
    Google Cloud Professional Data Engineer
    """
    
    # Create mock resume data
    mock_resume_data = {
        "full_text": sample_resume_text,
        "sections": {
            "experience": "Senior Data Scientist at TechCorp (2020-Present) - Developed machine learning models using Python, TensorFlow, and PyTorch - Worked with AWS for cloud deployment and Docker for containerization - Implemented data pipelines using Apache Spark and SQL - Created data visualizations using Tableau and Power BI - Led a team of 3 data scientists in various projects",
            "skills": "Programming: Python, R, SQL ML/DL: TensorFlow, PyTorch, Scikit-learn, Keras Cloud: AWS, Docker, Kubernetes Big Data: Spark, Hadoop Visualization: Tableau, Power BI, Matplotlib Other: Git, Jupyter, Linux",
            "education": "Master of Science in Data Science Stanford University",
            "certifications": "AWS Certified Machine Learning Specialist Google Cloud Professional Data Engineer"
        },
        "metadata": {
            "pages": 1,
            "char_count": len(sample_resume_text),
            "has_skills_section": True
        }
    }
    
    # Test skill extraction
    print("🧪 Testing complete skill extraction...")
    extracted_skills = intelligence_core.skill_extractor.extract(mock_resume_data, debug=True)
    user_skills = [s.normalized for s in extracted_skills]
    
    print(f"✅ Extracted {len(extracted_skills)} skills:")
    for skill in extracted_skills[:15]:
        print(f"   • {skill.original} (confidence: {skill.confidence:.2f}, source: {skill.source})")
    
    # Test matching for different roles
    test_roles = ["Data Scientist", "Software Engineer", "Product Manager", "Business Analyst"]
    
    print(f"\n🎯 Testing role matching:")
    for role in test_roles:
        job_profile = intelligence_core.job_skills_db.load_job_skills(role)
        
        match_score, confidence = intelligence_core.matching_engine.compute_match(
            user_skills, job_profile["tech_skills"], job_profile["soft_skills"]
        )
        
        print(f"   {role}: {match_score}% match (confidence: {confidence})")
    
    # Test gap analysis for best match
    best_role = "Data Scientist"
    job_profile = intelligence_core.job_skills_db.load_job_skills(best_role)
    
    all_job_skills = job_profile["tech_skills"] + job_profile["soft_skills"]
    missing_skills = [s for s in all_job_skills if s.lower() not in user_skills]
    transferable_skills = intelligence_core.matching_engine.find_transferable_skills(
        user_skills, missing_skills
    )
    
    missing_required, missing_preferred = intelligence_core.gap_analyzer.analyze(
        user_skills, job_profile["tech_skills"], job_profile["soft_skills"], transferable_skills
    )
    
    print(f"\n📈 Gap Analysis for {best_role}:")
    print(f"   Missing required: {len(missing_required)}")
    print(f"   Missing preferred: {len(missing_preferred)}")
    print(f"   Transferable skills: {len(transferable_skills)}")
    
    if missing_required:
        print(f"\n🔥 Top missing required skills:")
        for gap in missing_required[:3]:
            print(f"   • {gap.name} ({gap.priority}, {gap.estimated_weeks} weeks)")
    
    if transferable_skills:
        print(f"\n🔄 Transferable skills found:")
        for ts in transferable_skills[:3]:
            print(f"   • '{ts.user_skill}' → '{ts.maps_to_job_skill}' ({int(ts.transfer_score*100)}%)")
    
    return extracted_skills

# Run the complete test
print("🧪 Running complete system test...")
test_skills = test_complete_system()
print("\n✅ Complete system test finished!")
print("🎉 The Bridgr ML model is working correctly in Colab!")